In [5]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

In [6]:
np.random.seed(42)

In [8]:
# -----------------------
# CONFIG
# -----------------------
NUM_USERS = 10000
NUM_TRANSACTIONS = 200000

In [9]:
# -----------------------
# CITY-COUNTRY MAPPING
# -----------------------
city_country_map = {
    "India": ["Delhi", "Mumbai", "Indore", "Pune", "Jammu"],
    "Nepal": ["Kathmandu", "Pokhara"],
    "Afghanistan": ["Kabul", "Mazar-i-Sharif", "Herat", "Jalalabad"]
}

In [10]:
countries = list(city_country_map.keys())
country_probs = [0.7, 0.2, 0.1]

In [11]:
channels = ["ads", "organic", "referral"]
payment_methods = ["upi", "card", "wallet"]
failure_reasons = ["bank_declined", "insufficient_balance", "network_error", "timeout"]

In [12]:
# -----------------------
# USERS TABLE
# -----------------------
user_data = []

for user_id in range(1, NUM_USERS + 1):
    country = np.random.choice(countries, p=country_probs)
    city = np.random.choice(city_country_map[country])

    signup_date = pd.to_datetime("2022-01-01") + pd.to_timedelta(np.random.randint(0, 900), unit="D")

    user_data.append([
        user_id,
        signup_date,
        country,
        city,
        np.random.choice(channels, p=[0.5, 0.3, 0.2])
    ])

users = pd.DataFrame(user_data, columns=[
    "user_id", "signup_date", "country", "city", "acquisition_channel"
])

In [13]:
users

,user_id,signup_date,country,city,acquisition_channel
0,1,2022-09-28,India,Jammu,organic
1,2,2023-04-12,India,Mumbai,ads
2,3,2022-04-10,India,Jammu,ads
3,4,2024-02-09,India,Jammu,organic
4,5,2022-07-11,Afghanistan,Mazar-i-Sharif,referral
...,...,...,...,...,...
9995,9996,2023-01-27,India,Delhi,referral
9996,9997,2023-06-29,India,Indore,referral
9997,9998,2022-07-04,Afghanistan,Jalalabad,ads
9998,9999,2024-04-11,India,Jammu,referral


In [14]:
# -----------------------
# TRANSACTIONS TABLE
# -----------------------
transactions = pd.DataFrame({
    "transaction_id": range(1, NUM_TRANSACTIONS + 1),
    "user_id": np.random.randint(1, NUM_USERS + 1, NUM_TRANSACTIONS),
    "transaction_date": pd.to_datetime("2023-01-01") + pd.to_timedelta(np.random.randint(0, 365, NUM_TRANSACTIONS), unit="D"),
    "amount": np.round(np.random.exponential(scale=2000, size=NUM_TRANSACTIONS), 2),
    "status": np.random.choice(["success", "failed"], NUM_TRANSACTIONS, p=[0.9, 0.1]),
    "payment_method": np.random.choice(payment_methods, NUM_TRANSACTIONS, p=[0.6, 0.25, 0.15])
})

# -----------------------
# ADD SEASONALITY (Festival spike)
# -----------------------
transactions["month"] = transactions["transaction_date"].dt.month
transactions.loc[transactions["month"].isin([10, 11]), "amount"] *= 1.3

# -----------------------
# HIGH VALUE USERS (Top 5%)
# -----------------------
high_value_users = np.random.choice(users["user_id"], size=int(0.05 * NUM_USERS))

transactions.loc[
    transactions["user_id"].isin(high_value_users),
    "amount"
] *= 3

# -----------------------
# INCREASE FAILURE FOR UPI
# -----------------------
upi_mask = (transactions["payment_method"] == "upi") & (np.random.rand(NUM_TRANSACTIONS) < 0.18)
transactions.loc[upi_mask, "status"] = "failed"

In [15]:
transactions

,transaction_id,user_id,transaction_date,amount,status,payment_method,month
0,1,2172,2023-08-08,295.250,success,wallet,8
1,2,8348,2023-12-05,3126.610,success,card,12
2,3,6254,2023-06-24,2906.220,success,upi,6
3,4,9895,2023-07-05,2668.220,success,upi,7
4,5,3386,2023-12-13,1375.360,failed,upi,12
...,...,...,...,...,...,...,...
199995,199996,9265,2023-12-05,460.600,success,card,12
199996,199997,9072,2023-01-19,177.500,success,card,1
199997,199998,4836,2023-10-14,233.805,success,card,10
199998,199999,3195,2023-06-23,545.800,success,upi,6


In [16]:
# -----------------------
# PAYOUTS TABLE (ONLY SUCCESS)
# -----------------------
success_tx = transactions[transactions["status"] == "success"].copy()

payouts = pd.DataFrame({
    "payout_id": range(1, len(success_tx) + 1),
    "transaction_id": success_tx["transaction_id"],
    "payout_amount": success_tx["amount"] * np.random.uniform(0.90, 0.97, len(success_tx)),
    "commission_fee": success_tx["amount"] * np.random.uniform(0.03, 0.08, len(success_tx)),
    "payout_status": np.random.choice(["completed", "pending"], len(success_tx), p=[0.9, 0.1])
})

In [19]:
payouts

,payout_id,transaction_id,payout_amount,commission_fee,payout_status
0,1,1,276.304769,22.813573,completed
1,2,2,2917.480003,225.963622,completed
2,3,3,2797.169042,157.095608,completed
3,4,4,2518.708425,151.180134,completed
7,5,8,122.416020,4.664710,completed
...,...,...,...,...,...
199995,160717,199996,420.096640,20.381711,completed
199996,160718,199997,168.800142,12.827840,completed
199997,160719,199998,214.064610,7.269308,completed
199998,160720,199999,513.368841,20.072917,completed


In [17]:
# -----------------------
# FAILURE LOGS (ONLY FAILED)
# -----------------------
failed_tx = transactions[transactions["status"] == "failed"].copy()

failure_logs = pd.DataFrame({
    "failure_id": range(1, len(failed_tx) + 1),
    "transaction_id": failed_tx["transaction_id"],
    "failure_reason": np.random.choice(
        failure_reasons,
        len(failed_tx),
        p=[0.45, 0.30, 0.20, 0.05]  # biased
    ),
    "failure_time": failed_tx["transaction_date"] + pd.to_timedelta(np.random.randint(0, 60, len(failed_tx)), unit="m")
})

In [20]:
failure_logs

,failure_id,transaction_id,failure_reason,failure_time
4,1,5,bank_declined,2023-12-13 00:54:00
5,2,6,insufficient_balance,2023-01-05 00:20:00
6,3,7,bank_declined,2023-10-25 00:06:00
11,4,12,network_error,2023-12-15 00:45:00
14,5,15,bank_declined,2023-10-08 00:06:00
...,...,...,...,...
199964,39275,199965,insufficient_balance,2023-05-05 00:42:00
199967,39276,199968,bank_declined,2023-10-24 00:21:00
199970,39277,199971,insufficient_balance,2023-08-03 00:35:00
199978,39278,199979,bank_declined,2023-09-02 00:31:00


In [18]:
# -----------------------
# SAVE FILES
# -----------------------
users.to_csv("users.csv", index=False)
transactions.to_csv("transactions.csv", index=False)
payouts.to_csv("payouts.csv", index=False)
failure_logs.to_csv("failure_logs.csv", index=False)

print("✅ Data generated successfully")

✅ Data generated successfully


In [22]:
from google.colab import files
files.download('users.csv')
files.download('transactions.csv')
files.download('payouts.csv')
files.download('failure_logs.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>